# EUFOD Reference Manual: Core Theory & Mathematical Framework

This notebook serves as the theoretical reference for the **EUFOD Streamlit Application**. 

## 1. Physical Background & Chemical Principles

- **Lanthanide Shift Reagents (LSRs):** Paramagnetic coordination complexes (e.g., $\text{Eu(fod)}_3$) that coordinate Lewis-basic substrate functional groups (such as the -OH group in menthol).
- **Chemical Shift Dispersion:** Unpaired $4f$ electrons in $\text{Eu}^{3+}$ induce local magnetic fields, creating position-dependent chemical shift alterations ($\Delta\delta$) across the substrate.
- **Paramagnetic Dominance:** The total shift combines contact (through-bond) and pseudocontact (through-space) effects. For proton NMR in these complexes, the **pseudocontact shift** dominates and forms the basis of 3D spatial modeling.

## 2. Mathematical Model: McConnell-Robertson Equation

The pseudocontact shift $\Delta\delta_i$ for a given proton $i$ is defined as:

$$\Delta\delta_i = K \cdot G_i$$

where $G_i$ represents the geometric factor:

$$G_i = \frac{3\cos^2\theta_i - 1}{r_i^3}$$

### Variable Definitions
- **$r_i$**: Euclidean distance from the $\text{Eu}^{3+}$ center to proton $i$ ($|\mathbf{H}_i - \mathbf{Eu}|$).
- **$\theta_i$**: Angle between the coordination axis vector ($\mathbf{Eu} \rightarrow \mathbf{O}$) and the proton vector ($\mathbf{Eu} \rightarrow \mathbf{H}_i$).
- **$K$**: Temperature- and complex-dependent physical constant, uniform across all protons in the molecule.
- **Magic Angle:** At $\theta \approx 54.74^{\circ}$, $3\cos^2\theta - 1 = 0$, resulting in a net zero pseudocontact shift regardless of distance.

### Geometry

The McConnell–Robertson equation uses the Eu–H distance **r** and the angle **θ** between the Eu–O direction and the Eu–H vector.

![Eu–O–H Geometry](assets/eufod_geometry.svg)

## 3. Inverse Grid Search Strategy & Objective Metrics

The application solves an **inverse spatial problem**: given the molecular coordinates and experimental relative shifts, it searches for the position of $\mathrm{Eu}^{3+}$ that gives the best agreement between the calculated and experimental shift patterns.

### Search Volume

- **Search box:** A $6.0\text{ Å}$ Cartesian cube centered on the coordinating oxygen atom, corresponding to $\pm 3.0\text{ Å}$ along each Cartesian axis.
- **Grid resolution:** The grid step is configurable. Internally, coordinates are expressed in centiångströms (cÅ), so a step of `5` cÅ corresponds to $0.05\text{ Å}$.
- **Search procedure:** At every candidate Eu position, the McConnell–Robertson geometric term is calculated for all protons and normalized to obtain relative calculated shifts.

### Objective Metrics

EUFOD provides two alternative criteria for identifying the best Eu position.

**R-factor**

The default criterion measures the root-mean-square percentage error between relative shifts:

$$R = 100 \times \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i^{\mathrm{calc}} - y_i^{\mathrm{exp}}}{y_i^{\mathrm{exp}}} \right)^2}$$

Where $n$ is the total number of assigned protons. A lower $R$-factor indicates better agreement, and the grid point with the **minimum R-factor** is selected.

**Pearson correlation**

Alternatively, the similarity between the experimental and calculated shift patterns can be evaluated using the Pearson correlation coefficient:

$$r = \frac{\sum_i (y_i^{\mathrm{exp}}-\bar y^{\mathrm{exp}})(y_i^{\mathrm{calc}}-\bar y^{\mathrm{calc}})}{\sqrt{\sum_i (y_i^{\mathrm{exp}}-\bar y^{\mathrm{exp}})^2 \sum_i (y_i^{\mathrm{calc}}-\bar y^{\mathrm{calc}})^2}}$$

In this mode, the grid point with the **maximum Pearson correlation coefficient** is selected as the predicted Eu position.

Thus, depending on the selected metric, the optimal Eu coordinate $(x_E,y_E,z_E)$ is the point that either minimizes the percentage $R$-factor or maximizes the Pearson correlation.